In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [4]:
# set up
FILENAMEFREIGHT = 'merged_eurostat_clean_V2.csv'
FILENAMELAND = 'land_area_eu.csv'
FILENAMEPOP = 'population_eu.csv'
NONEU = ['BA', 'CH', 'LI', 'MK', 'NO', 'TR', 'UK', 'AL', 'IS', 'ME', 'RS']

datafreight = pd.read_csv(FILENAMEFREIGHT)
datafreight = datafreight[datafreight['TIME_PERIOD'] == 2023] # 2023 is the most recent year available for modal share data
datafreight = datafreight[~datafreight['geo'].isin(NONEU)] # remove non EU countries
datafreight = datafreight[['geo', 'TIME_PERIOD', 'Network_length_KM', 'Modal_Share_PCT', 'Observed_freight_total_THST']] # only keep relevant columns

dataland = pd.read_csv(FILENAMELAND)
dataland = dataland[['geo', 'OBS_VALUE']] # only keep relevant columns
dataland = dataland.rename(columns={'OBS_VALUE': 'land'})

datapop = pd.read_csv(FILENAMEPOP)
datapop = datapop.rename(columns={'geo: Geopolitical entity (reporting)': 'geo', 'OBS_VALUE: Observation value': 'population'}) # rename columns for easy handling
datapop['geo'] = datapop['geo'].str[:2] # only keep shorthand of geo
datapop = datapop[~datapop['geo'].isin(NONEU)] # remove non EU countries
datapop = datapop[datapop['age: Age class'] == 'TOTAL: Total']
datapop = datapop[['geo', 'population']] # only keep relevant columns

# merge dataframes into one and convert geo to iso3 code for use with plotly
datamerged = pd.merge(datafreight, dataland, on='geo', how='outer')
datamerged = pd.merge(datamerged, datapop, on='geo', how='outer')
datamerged['iso3'] = datamerged['geo'].replace({
    'AT': 'AUT',
    'BE': 'BEL',
    'BG': 'BGR',
    'HR': 'HRV',
    'CY': 'CYP',
    'CZ': 'CZE',
    'DK': 'DNK',
    'EE': 'EST',
    'FI': 'FIN',
    'FR': 'FRA',
    'DE': 'DEU',
    'EL': 'GRC',
    'HU': 'HUN',
    'IE': 'IRL',
    'IT': 'ITA',
    'LV': 'LVA',
    'LT': 'LTU',
    'LU': 'LUX',
    'MT': 'MLT',
    'NL': 'NLD',
    'PL': 'POL',
    'PT': 'PRT',
    'RO': 'ROU',
    'SK': 'SVK',
    'SI': 'SVN',
    'ES': 'ESP',
    'SE': 'SWE'
})
datamerged['Name'] = datamerged['geo'].replace({
    'AT': 'Austria',
    'BE': 'Belgium',
    'BG': 'Bulgaria',
    'HR': 'Croatia',
    'CY': 'Cyprus',
    'CZ': 'Czechia',
    'DK': 'Denmark',
    'EE': 'Estonia',
    'FI': 'Finland',
    'FR': 'France',
    'DE': 'Germany',
    'EL': 'Greece',
    'HU': 'Hungary',
    'IE': 'Ireland',
    'IT': 'Italy',
    'LV': 'Latvia',
    'LT': 'Lithuania',
    'LU': 'Luxembourg',
    'MT': 'Malta',
    'NL': 'Netherlands',
    'PL': 'Poland',
    'PT': 'Portugal',
    'RO': 'Romania',
    'SK': 'Slovakia',
    'SI': 'Slovenia',
    'ES': 'Spain',
    'SE': 'Sweden'
})
datamerged['Network_density'] = datamerged['Network_length_KM'] / datamerged['land']
datamerged['Network_density_pop'] = datamerged['Network_length_KM'] / datamerged['population']
datamerged.head(35)

,geo,TIME_PERIOD,Network_length_KM,Modal_Share_PCT,Observed_freight_total_THST,land,population,iso3,Name,Network_density,Network_density_pop
0,AT,2023.0,5577.000,29.3,92443.0,83882,9104772,AUT,Austria,0.066486,0.000613
1,BE,2023.0,3629.000,11.7,NaN,30667,11742796,BEL,Belgium,0.118336,0.000309
2,BG,2023.0,4029.000,19.2,17105.0,110996,6447710,BGR,Bulgaria,0.036299,0.000625
3,CZ,2023.0,9514.000,21.7,83120.0,78871,10827529,CZE,Czechia,0.120627,0.000879
4,DE,2023.0,38691.000,20.6,351819.0,357569,83118501,DEU,Germany,0.108206,0.000465
5,DK,2023.0,2448.000,8.2,6756.0,42925,5932654,DNK,Denmark,0.057030,0.000413
6,EE,2023.0,1171.000,20.4,10106.0,45336,1365884,EST,Estonia,0.025829,0.000857
7,EL,2023.0,1822.000,1.1,NaN,131694,10413982,GRC,Greece,0.013835,0.000175
8,ES,2023.0,16113.530,4.2,23890.0,505983,48085361,ESP,Spain,0.031846,0.000335
9,FI,2023.0,5915.000,22.0,27066.0,338363,5563970,FIN,Finland,0.017481,0.001063


In [5]:
# bar and map rail length
datamerged = datamerged.sort_values('Network_length_KM', ascending=False)
fig1 = px.bar(datamerged, 'geo', 'Network_length_KM', 
       labels={
           'geo': 'Country',
           'Network_length_KM': 'Network Length (KM)'
       },
       title='Network Length by Country')
fig1.update_xaxes(
    tickvals=datamerged['geo'],
    ticktext=datamerged['Name']
)
fig1.show()

fig2 = px.choropleth(
    datamerged,
    locations='iso3',     
    color='Network_length_KM',          
    color_continuous_scale='Blues',  
    range_color=(0, datamerged['Network_length_KM'].max()),
    scope='europe',        
    labels={'Network_length_KM': 'Network Length (KM)'},
    title='Network Length (KM)'
)

fig2.update_layout(width=800, height=600)
fig2.show()

# scatterplot modal share vs rail length, modal share vs rail density


In [6]:
# bar and map rail density land
datamerged = datamerged.sort_values('Network_density', ascending=False)
fig3 = px.bar(datamerged, 'geo', 'Network_density', 
       labels={
           'geo': 'Country',
           'Network_density': 'Network Density (KM/KM2)'
       },
       title='Network Density by Country',
       text='Name')
fig3.update_xaxes(
    tickvals=datamerged['geo'],
    ticktext=datamerged['Name']
)
fig3.show()

fig4 = px.choropleth(
    datamerged,
    locations='iso3',     
    color='Network_density',          
    color_continuous_scale='Reds',  
    range_color=(0, datamerged['Network_density'].max()),
    scope='europe',        
    labels={'Network_density': 'Network Density (KM/KM2)'},
    title='Network Density (KM/KM2)'
)

fig4.update_layout(width=800, height=600)
fig4.show()

In [7]:
# bar and map rail density population
datamerged = datamerged.sort_values('Network_density_pop', ascending=False)
fig5 = px.bar(datamerged, 'geo', 'Network_density_pop', 
       labels={
           'geo': 'Country',
           'Network_density_pop': 'Network Density (KM/population)'
       },
       title='Network Density by Country',
       text='Name')
fig5.update_xaxes(
    tickvals=datamerged['geo'],
    ticktext=datamerged['Name']
)
fig5.show()

fig6 = px.choropleth(
    datamerged,
    locations='iso3',     
    color='Network_density_pop',          
    color_continuous_scale='Purples',  
    range_color=(0, datamerged['Network_density_pop'].max()),
    scope='europe',        
    labels={'Network_density_pop': 'Network Density (KM/population)'},
    title='Network Density (KM/population)'
)

fig6.update_layout(width=800, height=600)
fig6.show()

In [8]:
# bar and map modal share
datamerged = datamerged.sort_values('Modal_Share_PCT', ascending=False)
fig7 = px.bar(datamerged, 'geo', 'Modal_Share_PCT', 
       labels={
           'geo': 'Country',
           'Modal_Share_PCT': 'Modal Share (%)'
       },
       title='Rail modal share by country',
       text='Name')
fig7.update_xaxes(
    tickvals=datamerged['geo'],
    ticktext=datamerged['Name']
)
fig7.show()

fig8 = px.choropleth(
    datamerged,
    locations='iso3',     
    color='Modal_Share_PCT',          
    color_continuous_scale='Greens',  
    range_color=(0, datamerged['Modal_Share_PCT'].max()),
    scope='europe',        
    labels={'Modal_Share_PCT': 'Modal_Share_PCT (%)'},
    title='Rail modal share by country'
)

fig8.update_layout(width=800, height=600)
fig8.show()

In [9]:
# scatterplots modal share
#   geo	TIME_PERIOD	Network_length_KM	Modal_Share_PCT	OBS_VALUE	iso3	Name	Network_density

fig9 = px.scatter(
    x=datamerged['Network_length_KM'], 
    y=datamerged['Modal_Share_PCT'],
    title='Relationship between Network Length and Modal Share',
    labels={
        'x': 'Network Length (KM)',
        'y': 'Modal Share (%)'
    },
    trendline='ols')

fig9.show()

fig10 = px.scatter(
    x=datamerged['Network_density'], 
    y=datamerged['Modal_Share_PCT'], 
    title='Relationship between Network Density (Land) and Modal Share',
    labels={
        'x': 'Network Density (KM/KM2)',
        'y': 'Modal Share (%)'
    },
    trendline='ols')
fig10.show()

fig11 = px.scatter(
    x=datamerged['Network_density_pop'], 
    y=datamerged['Modal_Share_PCT'], 
    title='Relationship between Network Density (Population) and Modal Share',
    labels={
        'x': 'Network Density (KM/Population)',
        'y': 'Modal Share (%)'
    },
    trendline='ols')
fig11.show()

In [10]:
# scatterplots total volume
#   geo	TIME_PERIOD	Network_length_KM	Modal_Share_PCT	OBS_VALUE	iso3	Name	Network_density

fig12 = px.scatter(
    x=datamerged['Network_length_KM'], 
    y=datamerged['Observed_freight_total_THST'],
    title='Relationship between Network Length and Total Freight Volume',
    labels={
        'x': 'Network Length (KM)',
        'y': 'Total Freight Volume (Tonnes)'
    },
    trendline='ols')

fig12.show()

fig13 = px.scatter(
    x=datamerged['Network_density'], 
    y=datamerged['Observed_freight_total_THST'], 
    title='Relationship between Network Density (Land) and Total Freight Volume',
    labels={
        'x': 'Network Density (KM/KM2)',
        'y': 'Total Freight Volume (Tonnes)'
    },
    trendline='ols')
fig13.show()

fig14 = px.scatter(
    x=datamerged['Network_density_pop'], 
    y=datamerged['Observed_freight_total_THST'], 
    title='Relationship between Network Density (Population) and Total Freight Volume',
    labels={
        'x': 'Network Density (KM/Population)',
        'y': 'Total Freight Volume (Tonnes)'
    },
    trendline='ols')
fig14.show()

In [15]:
# some testing (not for report)
datamerged['population_density'] = datamerged['population'] / datamerged['land']
datamerged = datamerged[datamerged['geo'] != 'MT']

fig15 = px.scatter(
    x=datamerged['population_density'], 
    y=datamerged['Modal_Share_PCT'], 
    title='Relationship between Population Density and Modal Share',
    labels={
        'x': 'Population Density (Population/KM2)',
        'y': 'Modal Share (%)'
    },
    trendline='ols')
fig15.show()